<a href="https://colab.research.google.com/github/nomanabdullah04/Natural-Language-Processing-NLP-/blob/main/Stemming_In_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## 2. The Core Concept Explained via Analogy

Imagine you are sorting a massive pile of tools in a workshop garage. You have a collection of screwdrivers: a **standard screwdriver**, a **long screwdriver**, a **stubby screwdriver**, and a **powered screwdriver**.

### The Morphological Dilemma
If your storage system forces you to create an entirely separate, unique labeled box for every variations of the tool, your inventory management system will break down due to clutter (Data Sparsity). You don't care about the specific size or handle type right now; you just need to know that they all perform the foundational action of **screwing**.

### The Stemming Action
Stemming acts like a crude pair of heavy-duty shop shears. Instead of carefully studying each tool's blueprint, it simply chops off the handles, grips, and extensions until only the core metal shaft is left.
* `screwdriver` $\rightarrow$ `screw`
* `screwed` $\rightarrow$ `screw`
* `screwing` $\rightarrow$ `screw`

The resulting item (`screw`) isn't a pristine, functional tool anymore, but it serves as a perfectly unified, low-space token that groups all similar tools together under one exact category.

---

## 3. Step-by-Step Mechanical Walkthrough (With String Examples)

The algorithm strips suffixes away by testing target string arrays against mathematical word structures (vowel-consonant sequences). Here is how a standard Porter Stemmer mechanically treats different structural words:

### Case 1: Standard Suffix Stripping (Normal Processing)
When the algorithm encounters regular inflections, it strips the trailing characters predictably:
* **Target Word:** `Replacing`
* **Mechanism:** The algorithm scans the ending, identifies the active particle `-ing`, checks that the preceding string `replac` contains a vowel, and slices the ending away.
* **Output Stem:** `replac`

### Case 2: Over-Stemming Failure (The Word Distortion Problem)
Because the rule-based engine does not have a real dictionary to read, it cannot differentiate between words that look alike but mean different things:
* **Target Words:** `Organization` and `Organ`
* **Mechanism:** The algorithm aggressively targets the long suffix `-ization` and slices it down to its baseline root.
* **Resulting Map:** Both words collapse identically into the exact same stem: `organ`.
* **The Error:** A medical database tracking *human organs* will now accidentally return search hits for *corporate organizations*, introducing massive data corruption.

### Case 3: Irregular Word Failure (The Under-Stemming Problem)
If a word changes its internal spelling when changing tenses, the suffix slicing engine fails entirely:
* **Target Words:** `Good`, `Better`, and `Best`
* **Mechanism:** The algorithm checks the endings for regular suffixes like `-ed` or `-ing`. Finding none, it leaves the strings completely untouched.
* **Resulting Map:** `good` $\rightarrow$ `good` | `better` $\rightarrow$ `better` | `best` $\rightarrow$ `best`
* **The Error:** The machine learning model treats these three words as completely independent features, failing to realize they share an identical semantic baseline.


In [ ]:
#Steming

words = ['Replacing', 'Organization', 'Organ', 'Good', 'Better', 'Best','eating','eaten','eat']


# **PorterStemmer**

## 1. Core Architecture & Mathematical Foundations

The **Porter Stemmer** is a deterministic, rule-based heuristic algorithm designed by Martin Porter in 1980 for removing the most common morphological and inflectional suffixes from English words. Rather than using an expensive, memory-heavy lookup dictionary (like Lemmatizers), it operates entirely via cascading conditional string-slicing rules.

### A. The Measure Metric ($m$)
To prevent stripping vital characters from short words, the Porter Stemmer maps every English word to a structural template of alternating consonant clusters ($C$) and vowel clusters ($V$). The complexity of a word is measured by its **Measure ($m$)**, defined as the number of $VC$ repetitions between the first vowel and the last consonant:

$$[\text{C}](\text{VC})^m[\text{V}]$$

* **$m = 0$:** `TR` ($C$), `EE` ($V$), `TREE` ($CVC \rightarrow m=0$)
* **$m = 1$:** `TROUBLE` ($VC \rightarrow m=1$)
* **$m = 2$:** `TROUBLES` ($VCVC \rightarrow m=2$)

### B. Cascading Conditional Rules
A standard Porter rule follows the syntax:
$$(\text{Condition}) \quad S_1 \rightarrow S_2$$
This dictates that if a word ends with suffix $S_1$, and the remaining stem meets the specified structural **Condition** (such as a minimum measure $m$), $S_1$ is replaced by $S_2$.

---

## 2. Core Functional Pipeline (The 5 Cascading Steps)

The algorithm operates through 5 distinct sequential functional blocks. A text token enters at Step 1 and passes down to Step 5, updating its internal string structure at each step.

[Raw Token] ──► [Step 1: Plurals & Tenses] ──► [Step 2: Vowel Ends] ──► [Step 3/4: Complex Suffixes] ──► [Step 5: E/L Removal] ──► [Final Stem]

###  Step 1: Plurality and Basic Verb Tenses
Handles basic inflectional endings like `-sses`, `-ies`, `-ed`, and `-ing`.
* **Rule Example:** `SSES -> SS` (e.g., *caresses* $\rightarrow$ *caress*)
* **Rule Example:** $(m > 0) \text{ EED -> EE}$ (e.g., *agreed* $\rightarrow$ *agree*, but *feed* stays *feed* because its base $m=0$).

### Step 2: Terminal Character Regularization
Turns terminal `-y` into `-i` if there is another vowel in the stem, creating a standard baseline for vowel transitions.
* **Rule Example:** $(\text{*v*}) \text{ Y -> I}$ (e.g., *happy* $\rightarrow$ *happi*, *sky* stays *sky*).

###  Step 3: Derivational Mapping (Double Suffix Stripping)
Looks at complex multi-layered word endings and maps them down to simpler configurations.
* **Rule Example:** $(m > 0) \text{ ATIONAL -> ATE}$ (e.g., *relational* $\rightarrow$ *relate*)
* **Rule Example:** $(m > 0) \text{ IZER -> IZE}$ (e.g., *digitizer* $\rightarrow$ *digitize*)

###  Step 4: Deletion of Strong Derivational Suffixes
Removes major structural suffixes to expose the naked baseline stem of the word.
* **Rule Example:** $(m > 1) \text{ AL -> }\emptyset$ (e.g., *revival* $\rightarrow$ *reviv*)
* **Rule Example:** $(m > 1) \text{ ANCE -> }\emptyset$ (e.g., *allowance* $\rightarrow$ *allow*)

###  Step 5: Cleansing & Compression Cleanup
Trims remaining trailing vowels or double consonants based on the final measure score.
* **Rule Example:** $(m > 1) \text{ E -> }\emptyset$ (e.g., *probate* $\rightarrow$ *probat*)
* **Rule Example:** $(m > 1 \text{ and end in double consonant L}) \text{ -> single L}$ (e.g., *controll* $\rightarrow$ *control*).


## 3. Concrete Scenario Examples

Here is exactly how the rule paths manipulate different words down to their final structural stems:

###  Example 1: Successful Rule Reduction (Normal Behavior)
* **Input Word:** `generalizations`
* **Step 1:** The plural `-s` matches $\rightarrow$ `generalization`
* **Step 3:** Matches `ATIONAL -> ATE` $\rightarrow$ `generalize`
* **Step 4:** Matches `IZE -> `$\emptyset$ $\rightarrow$ `general`
* **Output Stem:** `general` (Perfect mapping back to the semantic baseline root)

###  Example 2: Over-Stemming Failure (False Positive)
Because the algorithm knows rules instead of meanings, it over-chops valid word strings:
* **Input Words:** `organization` and `organ`
* **Processing:** `organization` matches Step 3/4 rules and aggressively truncates down to `organ`.
* **Output Stems:** Both words collapse identically into `organ`.
* **The Error:** A machine learning model will now evaluate *corporate organizations* and *biological human organs* as the exact same feature, muddying your data patterns.

###  Example 3: Under-Stemming Failure (False Negative)
If a word changes its internal spelling when shifting tenses or forms, Porter fails to group them:
* **Input Words:** `ran`, `runs`, and `running`
* **Processing:** `runs` $\rightarrow$ `run`, `running` $\rightarrow$ `run`. However, `ran` matches zero suffix rules and is left completely unmodified.
* **Output Stems:** `ran` stays `ran` | `runs` becomes `run`.
* **The Error:** The model treats `ran` and `run` as separate indices, completely missing the fact that they share an identical semantic action.

## 4. Summary of Strategic Trade-offs

###  Advantages (Pros)
* **Maximum Processing Velocity:** Operates via simple string adjustments, handling raw text significantly faster than database-heavy Lemmatizers.
* **Ultra-Low Memory Footprint:** Requires zero external dictionary files or semantic lexicons, making it ideal for edge servers or resource-constrained Colab runtimes.
* **Dimensionality Compression:** Natively reduces matrix sparsity by shrinking thousands of inflected words into a much smaller set of core features.

###  Disadvantages (Cons)
* **Loss of Readability:** Frequently produces non-words and fragments (e.g., *beauty* $\rightarrow$ *beauti*, *execute* $\rightarrow$ *execut*), making data dashboards hard for humans to audit.
* **Total Irregular Blind Spot:** Completely incapable of linking irregular verbs or nouns (e.g., *go/went*, *mouse/mice*) because it lacks semantic knowledge.
* **Strictly Monolingual:** Hardcoded entirely for the grammatical structures of the English language; fails completely if passed text in other languages.

In [ ]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

for word in words:
  print(word+"------->"+ps.stem(word))


Replacing------->replac
Organization------->organ
Organ------->organ
Good------->good
Better------->better
Best------->best
eating------->eat
eaten------->eaten
eat------->eat


In [ ]:
ps.stem('Congratulation')

'congratul'

In [ ]:
ps.stem('sitting')

'sit'

# ***RegexStemmerClass***

## 1. Core Architecture & Philosophy

The **RegexStemmer** (frequently implemented as `RegexpStemmer` in frameworks like NLTK) is a deterministic, custom-defined preprocessing component. Unlike the **Porter Stemmer**, which uses a fixed, complex, multi-stage cascading pipeline of standard English grammar rules, the RegexStemmer operates on a **user-defined programmable pattern matrix**.

It uses Regular Expressions (Regex) to target specific, predictable morphological structures (suffixes or prefixes) and instantly strip or replace them.

### A. The Structural Mapping Function
The RegexStemmer maps an incoming token stream through an explicit pattern configuration filter. For a target word $W$, the stemmer applies a regex substitution rule to yield the stem $S$:

$$f(W, \text{pattern}, \text{replacement}) \rightarrow S$$

$$\text{Example (Suffix Removal): } \text{pattern} = \text{"ing\$"}, \, \text{replacement} = \text{""}$$
$$\{\text{running, walking, cycling}\} \xrightarrow{f} \{\text{run, walk, cycl}\}$$

## 2. The Core Concept Explained via Analogy

Imagine you work at a shipping warehouse, and your job is to sort thousands of packages coming from different countries.

### The Fixed Tool Dilemma (Porter Stemmer)
If you use a automated machine hardcoded exclusively for **US Postal rules**, it will perfectly strip off state zip code tags. However, if a batch of international packages arrives with completely different tracking tag formats (slang, code bases, domain suffixes, or foreign languages), the machine breaks down because its hardcoded rules do not apply.

### The RegexStemmer Solution
The RegexStemmer acts like a **programmable label cutter**. You don't rely on pre-built national rules. Instead, you look at the incoming boxes, identify a specific recurring pattern yourself, and program the blade: *"Cut off exactly the last 3 characters if they read '.io'"* or *"Delete the prefix 'un-' at the start."*

It does not understand grammar, language, or vocabulary—it is a high-speed, precision cutting tool that cuts exactly where you tell it to.


## 3. Custom Rule Matrices & Functional Settings

Because you define the rules, the functional capability depends entirely on your regex configuration. The framework evaluates three structural parameters:
1. **The Pattern String:** The targeted regular expression string matching your prefix/suffix tokens.
2. **Min Length:** A safety counter variable. The algorithm will refuse to chop a word if the resulting stem length drops below this number (preventing short words like *king* from becoming *k*).
3. **The Replacement String:** Usually set to an empty string `""` to delete the pattern, but can be set to characters to regularize ends.

### Common Production Configuration Scenarios

| Targeted Text / Domain | Regex Pattern | Technical Objective |
| :--- | :--- | :--- |
| **Domain Name Truncation** | `r'(?:\.com\|\.org\|\.net)$'` | Strips web domain endings to group core brand names together. |
| **Hyper-Specific Verbal Cleaning** | `r'(?:ing\|ed\|s)$'` | A lightweight, fast alternative to full Porter stemming for basic English text. |
| **Medical / Technical Lexicons** | `r'(?:itis\|ology)$'` | Groups terms by core conditions or fields (e.g., *arthritis* $\rightarrow$ *arthr*, *biology* $\rightarrow$ *bi*). |



## 4. Concrete Operational Examples

Here is exactly how the RegexStemmer processes tokens under different custom pattern configurations:

###  Example 1: High-Precision Target Processing
* **Configured Pattern:** `r's$'` (Targeting trailing plural `s`), **Min Length:** `4`
* **Word A:** `bicycles` $\rightarrow$ Ends in `s`, length of stem `bicycle` (7) $\ge$ 4. **Output Stem:** `bicycle`
* **Word B:** `gas` $\rightarrow$ Ends in `s`, but length of stem `ga` (2) $<$ 4. **Rule Aborted.** **Output Stem:** `gas`
* **Result:** Successfully normalizes plurals without destroying critical short words.

###  Example 2: Over-Stemming Failure (False Positive)
Because the regex engine matches characters blindly without structural context, broad rules introduce errors:
* **Configured Pattern:** `r'ing$'` (Targeting standard present continuous verbs)
* **Input Words:** `running` and `king`
* **Processing:** `running` matches the pattern cleanly and outputs `run`. However, `king` also matches the literal characters `ing` at the end.
* **Output Stems:** `running` $\rightarrow$ `run` | `king` $\rightarrow$ `k`
* **The Error:** The noun `king` is completely corrupted into a single character token `k`, destroying its semantic utility in downstream machine learning vectors.

###  Example 3: Under-Stemming Failure (False Negative)
If your target text displays diverse variations outside your strict pattern list, it bypasses the system completely:
* **Configured Pattern:** `r'(?:ed\|ing)$'`
* **Input Words:** `walked`, `walking`, and `walks`
* **Processing:** `walked` $\rightarrow$ `walk`. `walking` $\rightarrow$ `walk`. However, `walks` ends in `s`, which does not match the regex array.
* **Output Stems:** `walked/walking` collapse to `walk`, but `walks` remains `walks`.
* **The Error:** Your vocabulary space still contains separate features for `walk` and `walks`, causing unwanted data sparsity.


## 5. Summary of Strategic Trade-offs

###  Advantages (Pros)
* **Absolute Domain Customization:** Allows you to build highly targeted text processing for domains where traditional English grammar models fail completely (e.g., medical tags, URLs, log files, programming code strings).
* **Maximum Execution Velocity:** Runs an optimized singular regular expression search-and-replace mechanism, outperforming multi-stage algorithmic pipelines.
* **Multilingual Capability:** Completely decoupled from English grammar constraints. You can write custom regex patterns for Spanish, French, or any structural language.

###  Disadvantages (Cons)
* **Zero Structural Flexibility:** If an inflection falls outside the exact characters declared in your regex string, the stemmer cannot adapt or process it.
* **High Operational Risk of Corruption:** Aggressive or poorly structured regex strings can accidentally wipe out base roots of words (as seen with *king* $\rightarrow$ *k*), heavily corrupting your feature space.
* **Manual Maintenance:** Demands continuous human optimization to update and audit pattern blocks as new text arrays enter the system.


In [ ]:
from nltk.stem import RegexpStemmer

#RegexpStemmer("expression",minimum words)

rs = RegexpStemmer('ing$|s$|ed$|tion$',min=6)

for word in words:
  print(word+"------->"+rs.stem(word))

Replacing------->Replac
Organization------->Organiza
Organ------->Organ
Good------->Good
Better------->Better
Best------->Best
eating------->eat
eaten------->eaten
eat------->eat


# ***Snowball Stemmer***

## 1. Core Architecture & Philosophy

The **Snowball Stemmer** (originally named the **Porter2 Stemmer**) is a deterministic, rule-based text normalization algorithm designed by Martin Porter. It was developed to resolve the structural design flaws and rigidity of the classic 1980 Porter Stemmer.

Rather than being written as a single, hardcoded software script, Snowball is built on a specialized algorithmic definition framework (the **Snowball string processing language**). This framework allows engineers to program custom, highly optimized rule cascades for multiple human languages, not just English.

### A. Algorithmic Enhancements over Classic Porter
The Snowball Stemmer modifies the traditional computational pipeline in three primary ways:
1. **Integrated Dictionary Exceptions:** Natively accommodates explicit exception lookup arrays for highly common irregular words before running structural rules.
2. **Aggressive Multi-Character Identification:** Evaluates combinations of vowels and consonants more accurately, preventing the slicing of vital character clusters.
3. **Caseless Normalization Blocks:** Optimizes character evaluations inside runtime threads, accelerating preprocessing loops.


## 2. The Core Concept Explained via Analogy

Imagine you are managing an international recycling plant that processes empty plastic bottles.

### The Classic Tool Dilemma (Porter Stemmer)
Your baseline machine (Classic Porter) has a fixed mechanical blade. It works well for standard 1-liter bottles, but if it encounters a bottle with a slightly thicker neck or a non-standard cap, it either jams or over-chops the bottle, turning recyclable material into useless trash. Furthermore, it completely breaks if you feed it international bottle designs.

### The Snowball Solution
The Snowball Stemmer is a **modular, heavy-duty processing rig**.
* **For English Bottles:** It features an updated, highly sensitive blade system that knows not to chop off vital components (e.g., it knows the difference between a decorative ridges and the actual cap).
* **For International Processing:** You can instantly switch out the mechanical layout cartridge entirely. If a batch of Spanish or French bottles arrives, you drop in the corresponding language template, and the machine adapts its cutting strokes perfectly to the structural features of those specific items.



## 3. Structural Mechanics & Language Matrix

The algorithm evaluates words by establishing dynamic region boundaries within a string (known as **$R_1$** and **$R_2$**), which dictate where suffixes are safely allowed to be sliced without destroying the root.

$$\text{Word Layout: } [\text{Root}] \underbrace{\rule{1.5cm}{0.4pt}}_{R_1 \text{ Region}} \underbrace{\rule{1.5cm}{0.4pt}}_{R_2 \text{ Region}}$$

* **$R_1$ Region:** The structural zone of a word following the first consonant that comes immediately after a vowel.
* **$R_2$ Region:** The structural zone inside $R_1$ following the next consonant that comes after a vowel.
* **The Slicing Rule:** Suffix transformations are strictly forbidden unless the targeted characters fall completely within these verified, safe suffix regions.

### Global Language Distribution Matrix

| Selected Language | Algorithmic Custom Behavior | Impact on Feature Space |
| :--- | :--- | :--- |
| **English (Porter2)** | Special handling of terminal `-li` suffixes and specific vowel-consonant tracking. | **Highly Optimized.** Significantly fewer over-stemming errors compared to classic Porter. |
| **Spanish** | Targets complex romance verb inflections (`-ando`, `-iendo`, `-aremos`). | **High Compression.** Collapses vast grammatical verb tenses into tight data matrices. |
| **German** | Dynamically treats specific umlaut characters (`ä`, `ö`, `ü` $\rightarrow$ `ae`, `oe`, `ue`). | **Structural Normalization.** Groups compound variants cleanly. |



## 4. Concrete Operational Examples

Here is exactly how the Snowball Stemmer processes tokens compared to classic Porter adjustments:

###  Example 1: Successful Structural Optimization
* **Input Word:** `fairly`
* **Classic Porter:** Identifies `-ly` and blindly slices it down $\rightarrow$ `fairli`
* **Snowball (Porter2):** Checks the special structural rules for terminal `-li` endings, notices the preceding character constraints, and removes the suffix safely.
* **Output Stem:** `fair` (Maintains complete semantic mapping accuracy)

###  Example 2: Over-Stemming Failure (False Positive)
Despite its updates, Snowball is still a rule-based engine and can occasionally over-truncate strings:
* **Input Words:** `execute`, `execution`, and `executive`
* **Processing:** The cascading rules target the strong derivational endings inside the $R_1$/$R_2$ boundaries and slice them away.
* **Output Stems:** All three distinct words collapse identically into `execut`.
* **The Error:** A business analytics model will now treat an *executive manager* and the actions of *executing a server loop* as the exact same data vector feature, introducing data noise.

###  Example 3: Irregular Word Escape (Under-Stemming)
If an irregular form falls outside its hardcoded exception array, rule-based constraints will bypass it:
* **Input Words:** `go`, `went`, and `gone`
* **Processing:** The words are passed through the pipeline. Finding no regular suffix characters to strip, the strings pass through untouched.
* **Output Stems:** `go` $\rightarrow$ `go` | `went` $\rightarrow$ `went` | `gone` $\rightarrow$ `gone`
* **The Error:** Your model tracks these three terms as independent coordinates, completely missing the underlying semantic action relationship.



## 5. Summary of Strategic Trade-offs

###  Advantages (Pros)
* **Native Multi-Language Support:** Operates seamlessly across a wide matrix of European languages via specialized architectural rules.
* **Improved Linguistic Precision:** Significantly reduces classic over-stemming anomalies on English text blocks compared to the 1980 baseline.
* **High Computation Speed:** Retains the high-velocity string manipulation behavior of rule-based systems, outperforming database-reliant Lemmatizers.

###  Disadvantages (Cons)
* **Retains Nonsensical Fragments:** Outputs can still result in broken, non-dictionary words (e.g., `flies` $\rightarrow$ `fli`), creating visual parsing hurdles for human review.
* **No Contextual/POS Awareness:** Evaluates words completely in isolation; it cannot use Part-of-Speech tags to alter its behavior based on a word's usage in a sentence.
* **Increased System Complexity:** Managing multiple language configurations demands distinct pipeline initialization steps within your machine learning environment.


In [ ]:
from nltk.stem import SnowballStemmer
snb=SnowballStemmer('english')

for word in words:
  print(word+"------->"+snb.stem(word))

Replacing------->replac
Organization------->organ
Organ------->organ
Good------->good
Better------->better
Best------->best
eating------->eat
eaten------->eaten
eat------->eat


In [ ]:
snb.stem('fairly'),snb.stem('fairness')

('fair', 'fair')

# ***WorldNet Lemmatizer***
## 1. Core Architecture & Mathematical Foundations

Unlike rule-based stemmers that crudely slice off prefixes and suffixes, the **WordNet Lemmatizer** is a high-precision linguistic normalization engine. It maps inflected word forms back to their statistically valid base or dictionary form, known as the **Lemma**.

It achieves this by cross-referencing tokens against **WordNet**, a massive, structurally mapped lexical database of the English language developed by Princeton University.

### A. The Lemmatization Mapping Function
The WordNet Lemmatizer acts as a context-aware mapping function $f: (W, \text{POS}) \rightarrow L$, where a word token $W$ and its corresponding Part-of-Speech tag ($\text{POS}$) are mathematically resolved to a true dictionary lemma $L$:

$$f(W, \text{POS}) \rightarrow L$$

$$\text{Example (Verb Context): } f(\text{"went"}, \text{VERB}) \rightarrow \text{"go"}$$
$$\text{Example (Noun Context): } f(\text{"mice"}, \text{NOUN}) \rightarrow \text{"mouse"}$$

### B. Morphy: NLTK's Structural Search Engine
Underneath the implementation interface, the algorithm utilizes a built-in morphological processing framework called **Morphy**. When a word is ingested, Morphy triggers a two-tiered validation loop:
1. **The Index Check:** It immediately looks up the word in the WordNet database. If the word exists as a valid base entry, it is returned instantly.
2. **Detachment Rules + Exception Lists:** If the word is missing, Morphy applies clean suffix-detachment rules (e.g., `-ies` $\rightarrow$ `-y`) and tests the new variant against internal exception lists until a valid root structural connection is located in the lexicon.


## 2. The Core Concept Explained via Analogy

Imagine you are a strict librarian auditing a massive, unstructured archive of returned library books.

### The Crude Tool Approach (Stemming)
If you use a basic pair of shears (a Stemmer), you will blindly slice off the ends of book covers to fit them into uniform storage boxes. A book titled *"Running"* and a book titled *"Runners"* both get chopped down to read *"Run"*. This groups them together, but you have physically ruined the words, and if you encounter a book titled *"Went"*, your shears can do nothing to link it back to the original *"Go"* series.

### The WordNet Lemmatizer Solution
The WordNet Lemmatizer acts like a **scholarly research assistant with an encyclopedia**. Instead of cutting anything, the assistant looks closely at each book cover:
* If it reads *"mice"*, the assistant flips through the encyclopedia, identifies the structural relationship, and logs it under the master category **"mouse"**.
* If it reads *"went"*, the assistant safely routes it to the **"go"** section.

Nothing is broken, no words are mutilated into nonsense syllables, and every term is resolved to a real, grammatically correct entry. However, the assistant *must* be told whether the word is acting as a noun, verb, or adjective, or it will default to reading it as a standard noun.


## 3. The Power of Part-of-Speech (POS) Tagging

The WordNet Lemmatizer relies heavily on POS contexts. Because English words are often identical in spelling but completely different in grammatical usage (polysemy), passing a static word without context forces the lemmatizer to apply a default **Noun** filter.

$$\text{Raw Token Matrix: } \text{"saw"}$$
* $$f(\text{"saw"}, \text{NOUN}) \rightarrow \text{"saw"} \quad \text{(The cutting tool structures)}$$
* $$f(\text{"saw"}, \text{VERB}) \rightarrow \text{"see"} \quad \text{(The past-tense action of viewing)}$$

### POS Tag Mapping Matrix

| Text Token | Intended POS Tag | WordNet Constant | Resulting Lemma ($L$) | Behavioral Impact |
| :--- | :--- | :--- | :--- | :--- |
| **`leaves`** | Noun | `wordnet.NOUN` (`'n'`) | **`leaf`** | Cleans plurals flawlessly. |
| **`leaves`** | Verb | `wordnet.VERB` (`'v'`) | **`leave`** | Resolves active operational tenses. |
| **`better`** | Adjective | `wordnet.ADJ` (`'a'`) | **`good`** | Maps comparative scales back to origin. |


## 4. Concrete Operational Examples

Here is exactly how the WordNet Lemmatizer executes text normalization compared to standard rule-based truncation models:

###  Example 1: Successful Irregular Resolution
* **Input Tokens:** `was`, `is`, `am`, `been`
* **Stemmer Processing:** Left completely unchanged or clipped into nonsense roots due to lack of regular suffixes.
* **WordNet Lemmatizer (with VERB tag):** Maps every distinct variant flawlessly to the absolute dictionary root.
* **Output Lemma:** `be` (Perfect vector space consolidation)

###  Example 2: Out-of-Vocabulary (OOV) Escape
Because it relies on a strict dictionary, it cannot process modern internet slang, new technical terms, or typos:
* **Input Words:** `de-risking`, `upvoted`, and `googling`
* **Processing:** The engine scans the WordNet database. Finding no recorded historical dictionary match for these modern technical phrases, the detachment rules fail.
* **Output Lemmas:** `de-risking` $\rightarrow$ `de-risking` | `upvoted` $\rightarrow$ `upvoted`
* **The Error:** Fails to compress the feature dimensions, leaving your machine learning dataset sparse when handling modern colloquial data.

###  Example 3: Untagged Noun-Default Failure
If your pipeline fails to dynamically generate and pass POS tags, the system defaults to checking noun tables:
* **Input Words:** `spoken`, `sang`, and `flew`
* **Processing:** The engine assumes every token is a noun. It scans the noun dictionary for an entry matching those exact words. Finding no noun exceptions, it leaves them completely untouched.
* **Output Lemmas:** `spoken` $\rightarrow$ `spoken` | `sang` $\rightarrow$ `sang`
* **The Error:** Completely acts as a blind spot for verbs, losing all the compression benefits of lemmatization.


## 5. Summary of Strategic Trade-offs

###  Advantages (Pros)
* **Preserves Complete Readability:** Natively outputs real, grammatically correct dictionary words, ensuring clean, human-auditable data visualization dashboards.
* **Resolves Irregular Morphologies:** Easily connects completely different looking words (e.g., *went* $\rightarrow$ *go*, *worst* $\rightarrow$ *bad*) that absolutely break rule-based stemmers.
* **Highly Accurate Feature Vectors:** Eliminates the risk of over-stemming, protecting your text index from corrupting semantic relationships.

###  Disadvantages (Cons)
* **High Memory & Storage Footprint:** Requires loading the substantial WordNet structural database into your execution runtime environment memory space.
* **Significant Computational Latency:** Querying a large lookup database structure is substantially slower than executing basic regular expression or string-slicing rules.
* **Dependent on Pre-Tagging Pipelines:** Demands a secondary, upstream Part-of-Speech tagger pipeline to achieve maximum linguistic accuracy, adding workflow overhead.


In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from nltk.stem import WordNetLemmatizer
wnl=WordNetLemmatizer()

for word in words:
  print(word+"------->"+wnl.lemmatize(word,pos='v'))

Replacing------->Replacing
Organization------->Organization
Organ------->Organ
Good------->Good
Better------->Better
Best------->Best
eating------->eat
eaten------->eat
eat------->eat


### Understanding Part-of-Speech (POS) Tagging with WordNetLemmatizer

The `WordNetLemmatizer` is powerful but requires context, specifically the **Part-of-Speech (POS) tag**, to accurately lemmatize words.

*   When `pos='v'` (verb) is used, the lemmatizer tries to find the base form of the word as a verb.
*   When `pos='n'` (noun) is used, it looks for the base form as a noun.

If the wrong POS tag is provided, or if no tag is provided (which defaults to 'n' for noun), the lemmatizer might not produce the expected result, especially for words that have different forms or meanings depending on their use (e.g., 'Replacing' as a verb vs. 'organization' as a noun).

In [ ]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

for word in words:
  print(word+"------->"+wnl.lemmatize(word,pos='v'))

Replacing------->Replacing
Organization------->Organization
Organ------->Organ
Good------->Good
Better------->Better
Best------->Best
eating------->eat
eaten------->eat
eat------->eat
